In [1]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.5 MB/s eta 0:00:00


In [2]:
import torch
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))

cuda available: True
device: Tesla T4


In [3]:
!mkdir -p data

In [4]:
import json
import random

random.seed(42)

# The target schema the model should always emit:
# {"summary": str, "sentiment": "positive"|"negative"|"neutral",
#  "topics": [str], "action_items": [str]}
#
# The whole point: base model rambles or half-follows the format when you just
# ask in a prompt. After fine-tuning it locks to this exact JSON every time.

positive_texts = [
    ("The new onboarding flow shipped last Tuesday and signups are already up 30%. "
     "The team pulled it off ahead of schedule and the design reviews were smooth.",
     "Onboarding flow shipped early and signups rose 30%.",
     ["onboarding", "growth", "product"],
     ["Monitor signup numbers over the next month", "Share design review notes with the team"]),
    ("Really happy with how the Q3 offsite went. Everyone was engaged, the workshops "
     "landed well, and we left with a clear roadmap for next quarter.",
     "Q3 offsite was engaging and produced a clear roadmap.",
     ["offsite", "planning", "team"],
     ["Circulate the finalized roadmap", "Book the Q4 offsite venue"]),
    ("Customer support resolved the billing outage in under two hours and followed up "
     "with every affected account. Feedback has been overwhelmingly positive.",
     "Billing outage resolved quickly with positive customer feedback.",
     ["support", "billing", "reliability"],
     ["Write a postmortem for the outage", "Thank the support team publicly"]),
    ("The mentorship program hit its first cohort milestone. Mentees reported real "
     "progress and several mentors want to sign up again next round.",
     "Mentorship program's first cohort was a success.",
     ["mentorship", "learning", "community"],
     ["Open signups for the next cohort", "Collect testimonials from mentees"]),
    ("Our open-source library crossed 5k stars this week. Contributions are coming in "
     "steadily and the docs overhaul made a visible difference.",
     "Open-source library passed 5k stars with steady contributions.",
     ["open-source", "community", "documentation"],
     ["Highlight top contributors in the changelog", "Plan the next docs sprint"]),
]

negative_texts = [
    ("The deployment failed twice overnight and we still don't know the root cause. "
     "On-call was paged three times and the dashboard is showing elevated error rates.",
     "Repeated overnight deployment failures with unknown root cause.",
     ["deployment", "incident", "reliability"],
     ["Investigate the root cause", "Review on-call escalation load"]),
    ("Sales missed target for the second straight month. The pipeline looks thin and "
     "a couple of key deals slipped to next quarter.",
     "Sales missed target again with a thin pipeline.",
     ["sales", "revenue", "pipeline"],
     ["Audit the current pipeline", "Follow up on slipped deals"]),
    ("The migration corrupted a chunk of user records and the rollback took longer than "
     "expected. Several customers noticed missing data.",
     "Migration corrupted records and rollback was slow.",
     ["migration", "data", "incident"],
     ["Restore affected user records", "Add validation before the next migration"]),
    ("Morale on the platform team is low after the last reorg. People feel unclear on "
     "ownership and a few strong engineers have started interviewing elsewhere.",
     "Low morale on the platform team after the reorg.",
     ["morale", "reorg", "retention"],
     ["Clarify team ownership", "Schedule 1:1s with at-risk engineers"]),
    ("The vendor raised prices 40% with two weeks notice and their support has gotten "
     "noticeably worse. We're locked in until the contract renews.",
     "Vendor hiked prices sharply while support declined.",
     ["vendor", "cost", "contract"],
     ["Evaluate alternative vendors", "Flag the renewal date to finance"]),
]

neutral_texts = [
    ("The weekly metrics report is attached. Traffic held roughly flat, latency is "
     "within normal range, and no incidents were logged this week.",
     "Weekly metrics were stable with no incidents.",
     ["metrics", "monitoring"],
     ["File the report in the shared drive"]),
    ("We're switching the standup from 10am to 9:30am starting Monday to accommodate "
     "the new timezone spread on the team.",
     "Standup time moves to 9:30am on Monday.",
     ["scheduling", "team"],
     ["Update the calendar invite", "Notify the wider team"]),
    ("The API now supports pagination on the search endpoint. Existing clients are "
     "unaffected; the new parameters are optional.",
     "Search endpoint added optional pagination.",
     ["api", "search"],
     ["Update the API docs", "Add pagination examples"]),
    ("Inventory counts for the warehouse are complete. Numbers match the system of "
     "record with a small variance on two SKUs under review.",
     "Warehouse inventory counted with minor variance.",
     ["inventory", "operations"],
     ["Review the two flagged SKUs"]),
    ("The design system got a minor version bump. Spacing tokens were renamed; a "
     "codemod is available for teams that need to migrate.",
     "Design system minor bump renamed spacing tokens.",
     ["design-system", "tooling"],
     ["Run the codemod where needed", "Skim the migration notes"]),
]

instruction_templates = [
    "Read the following text and return the structured summary as JSON.\n\n{text}",
    "Extract the key details from this into the JSON schema.\n\n{text}",
    "Analyze the text below and give me the structured output.\n\n{text}",
    "Turn this into the standard JSON format.\n\n{text}",
    "Here's a note. Summarize it into the schema.\n\n{text}",
    "Process this text and respond with the JSON.\n\n{text}",
]

def make_examples():
    rows = []
    buckets = [
        ("positive", positive_texts),
        ("negative", negative_texts),
        ("neutral", neutral_texts),
    ]
    # repeat each source text across a few instruction phrasings for variety
    for sentiment, texts in buckets:
        for text, summary, topics, actions in texts:
            for tmpl in random.sample(instruction_templates, 4):
                instruction = tmpl.format(text=text)
                response = json.dumps({
                    "summary": summary,
                    "sentiment": sentiment,
                    "topics": topics,
                    "action_items": actions,
                }, ensure_ascii=False)
                rows.append({"instruction": instruction, "response": response})
    random.shuffle(rows)
    return rows

rows = make_examples()
with open("data/schema_sft.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(rows)} examples")

wrote 60 examples


In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/schema_sft.jsonl", split="train")
print(dataset)
print(dataset[0]["instruction"][:150])
print(dataset[0]["response"])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'response'],
    num_rows: 60
})
Read the following text and return the structured summary as JSON.

Customer support resolved the billing outage in under two hours and followed up wi
{"summary": "Billing outage resolved quickly with positive customer feedback.", "sentiment": "positive", "topics": ["support", "billing", "reliability"], "action_items": ["Write a postmortem for the outage", "Thank the support team publicly"]}


In [7]:
def to_chat(example):
    return {"messages": [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]}

dataset = dataset.map(to_chat, remove_columns=dataset.column_names)
dataset[0]["messages"]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

[{'content': 'Read the following text and return the structured summary as JSON.\n\nCustomer support resolved the billing outage in under two hours and followed up with every affected account. Feedback has been overwhelmingly positive.',
  'role': 'user'},
 {'content': '{"summary": "Billing outage resolved quickly with positive customer feedback.", "sentiment": "positive", "topics": ["support", "billing", "reliability"], "action_items": ["Write a postmortem for the outage", "Thank the support team publicly"]}',
  'role': 'assistant'}]

In [8]:
from google.colab import files
files.download("data/schema_sft.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("loaded")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

loaded


In [11]:
test_note = ("The payments service went down for 40 minutes during peak hours. "
             "Engineering traced it to a bad config push and rolled back. "
             "A few hundred transactions failed and will need manual retry.")

def generate(model, note, max_new_tokens=200):
    messages = [{"role": "user",
                 "content": f"Process this text and respond with the JSON.\n\n{note}"}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
        return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(generate(model, test_note))

{
  "status": "success",
  "message": "Payments service went down for 40 minutes during peak hours.",
  "details": {
    "config_push_bad_config": "A few hundred transactions failed and will need manual retry."
  }
}


In [13]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
print("lora config ready")

lora config ready


In [14]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qlora-schema-out",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
5,2.811146
10,1.563984
15,0.999092
20,0.708281
25,0.462207
30,0.300408
35,0.193740
40,0.150385


TrainOutput(global_step=40, training_loss=0.8986554130911827, metrics={'train_runtime': 93.7164, 'train_samples_per_second': 3.201, 'train_steps_per_second': 0.427, 'total_flos': 82519873067520.0, 'train_loss': 0.8986554130911827, 'epoch': 5.0})

In [16]:
trainer.model.config.use_cache = True
trainer.model.eval()

print(generate(trainer.model, test_note))

{"summary": "Payments service crashed during peak hours and config pushes were the cause.", "sentiment": "negative", "topics": ["payment-service", "fraud"], "action_items": ["Restore API aftercare", "Add validation in config pushes"]}


In [17]:
trainer.model.config.use_cache = True
trainer.save_model("qlora-schema-adapter")
tokenizer.save_pretrained("qlora-schema-adapter")
!ls -lh qlora-schema-adapter

total 28M
-rw-r--r-- 1 root root 1.2K Sep 23 19:41 adapter_config.json
-rw------- 1 root root  17M Sep 23 19:41 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Sep 23 19:41 chat_template.jinja
-rw-r--r-- 1 root root 5.1K Sep 23 19:41 README.md
-rw-r--r-- 1 root root  694 Sep 23 19:41 tokenizer_config.json
-rw-r--r-- 1 root root  11M Sep 23 19:41 tokenizer.json
-rw-r--r-- 1 root root 5.6K Sep 23 19:41 training_args.bin


In [18]:
!zip -r qlora-schema-adapter.zip qlora-schema-adapter
from google.colab import files
files.download("qlora-schema-adapter.zip")

  adding: qlora-schema-adapter/ (stored 0%)
  adding: qlora-schema-adapter/chat_template.jinja (deflated 71%)
  adding: qlora-schema-adapter/tokenizer_config.json (deflated 59%)
  adding: qlora-schema-adapter/adapter_model.safetensors (deflated 21%)
  adding: qlora-schema-adapter/training_args.bin (deflated 53%)
  adding: qlora-schema-adapter/tokenizer.json (deflated 81%)
  adding: qlora-schema-adapter/adapter_config.json (deflated 61%)
  adding: qlora-schema-adapter/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>